### Call Graph

In [5]:
# Necessary imports
import json
from pprint import pprint
from scalpel.call_graph.pycg import CallGraphGenerator, formats

In [7]:
# Gọi hàm analyze của lớp CallGraphGenerator:
cg_generator = CallGraphGenerator(["./test_repo/operations/add.py"], "./test_repo")
cg_generator.analyze()

In [ ]:
# Lấy output của đồ thị gọi: dict(node -> set(node))
cg = cg_generator.output()
pprint(cg)

In [ ]:
# Lấy danh sách các cạnh của đồ thị gọi: list((node, node))
edges = cg_generator.output_edges()
edges

In [ ]:
# Xác định vị trí (dòng bắt đầu và kết thúc) của các node:
internal_mods = cg_generator.output_internal_mods()
pprint(internal_mods)

In [ ]:
# Uhhh chắc là cái này không cần quan tâm lắm :/
external_mods = cg_generator.output_external_mods()
pprint(external_mods)

In [ ]:
# Format lại output: chuyển từ set thành list
formatter = formats.Simple(cg_generator)
pprint(formatter.generate())

# Thích thì export ra json file
with open("example_results.json", "w+") as f:
    f.write(json.dumps(formatter.generate()))

### Import Graph

In [1]:
# Necessary imports
from scalpel.import_graph.import_graph import ImportGraph, Tree

In [2]:
# Xây dựng cây Import từ thư mục test_repo
target_dir = "./test_repo"
import_graph = ImportGraph(target_dir)
import_graph.build_dir_tree()

In [ ]:
# Lấy các node con (module) và danh sách các phụ thuộc (imports) của nó
all_leaf_nodes = import_graph.get_leaf_nodes()
for node in all_leaf_nodes:
    module_dict = import_graph.parse_import(node.ast)
    print(node.name, module_dict)

### Control Flow Graph

In [14]:
# Necessary imports
import sys, ast
from scalpel.cfg import CFGBuilder, CFG

In [41]:
# Build CFG from file
cfg: CFG = CFGBuilder().build_from_file("SomeCFG", "./recurso.py")
function_cfgs = cfg.functioncfgs.items()

In [ ]:
# Build image from CFG
cfg.build_visual('png')

In [ ]:
# Build image from a CFG's function
funa_cfg = None 
for (block_id, fun_name), fun_cfg in function_cfgs:
    if fun_name == "fun_a":
        funa_cfg = fun_cfg
funa_cfg.build_visual('png')

In [ ]:
# Lấy ra các blocks (khối lệnh) từ CFG, mỗi blocks gồm một hoặc nhiều stmt
all_blocks = cfg.get_all_blocks()
entry_block = cfg.entryblock

In [ ]:
# Print all blocks (their id), source code, ...
print('\n'.join(map(str, all_blocks)))
print(f'\n{entry_block.get_source()}')

statements = entry_block.statements
for stmt in statements:
    print(ast.dump(stmt))

### Code Rewriter:

In [4]:
# Necessary imports
import ast
import os
import sys
import astor
from scalpel.rewriter import Rewriter

In [ ]:
# Định nghĩa rewrite rules
def rewrite_rules(node) -> list:
    org = node
    if not isinstance(node, ast.Expr):
        return [org]
    
    node = node.value
    if not isinstance(node, ast.Call) \
    or not hasattr(node.func, 'id') or node.func.id != 'print' \
    or len(node.args) > 1 or not isinstance(node.args[0].value, str):
        return [org]

    return [ast.Expr(ast.Call(
        func=ast.Name('print'),
        args=[
            ast.Constant(substr)
            for substr in node.args[0].value.split(' ')
        ],
        keywords=[]
    ))]


In [ ]:
# Mã nguồn cần test
src = """
def a():
    
"""

In [ ]:
# Áp dụng rewrite rules và print mã nguồn mới
rewriter = Rewriter(src)
new_src = rewriter.rewrite(src, rule_func=rewrite_rules)
print(new_src)

### Type Inference

In [9]:
# Necessary imports
from pprint import pprint
from scalpel.typeinfer.typeinfer import TypeInference

In [ ]:
# Áp dụng TypeInference lên file recurso.py
inferer = TypeInference(
    name="recurso.py", entry_point="./recurso.py"
)
inferer.infer_types()
inferred = inferer.get_types()

In [ ]:
# In ra danh sách kiểu dữ liệu dự đoán cho từng biến, hàm, tham số
pprint(inferred, width=60)

### Static Single Assignment (TODO: haven't understand it well :/)

In [ ]:
# Necessary imports
import os
import sys
import ast
import astor
import unittest

from scalpel.SSA.const import SSA
from scalpel.cfg import CFGBuilder, CFG

In [28]:
# Code thử nghiệm
code_str = """
b = 10
if b>0:
    a = b
else:
    a = 20
print(a)
"""

In [ ]:
cfg = CFGBuilder().build_from_src('test', code_str)
cfg.build_visual('png')

In [34]:
m_ssa = SSA()
ssa_results, const_dict = m_ssa.compute_SSA(cfg)

In [ ]:
for block_id, stmt_res in ssa_results.items():
    print(f"Result for block {block_id}:")
    print(stmt_res)

In [ ]:
for name, value in const_dict.items():
    print(name, value)

In [ ]:
for name, value in const_dict.items():
    if isinstance(value, ast.Name):
        print(name, value.id)

### API Name Qualifying

In [ ]:
# Necessary imports
import os
from pprint import pprint
from scalpel.core.mnode import MNode

In [71]:
# Code thử nghiệm
source = """
import numpy as np
import pandas as pd
from random import choices

def she_rejected_me():
    print(":sob:")

she_rejected_me()
she_rejected_me()

pd.read_csv("test.csv")
np.array([1,2,3,4,5,6])
data = [41, 50, 29]
means = sorted(mean(choices(data, k=len(data))) for i in range(100))
"""

In [72]:
# Đọc code, parse danh sách các hàm được gọi, parse các imports
mnode = MNode("local")
mnode.source = source
mnode.gen_ast()

func_calls = mnode.parse_func_calls()
import_dict = mnode.parse_import_stmts()

In [ ]:
# Parse danh sách các hàm được gọi và các imports
pprint(func_calls[0].keys()) # choices đâu ??????
print(func_calls[0].values())
pprint([f['name'] for f in func_calls])
pprint(import_dict)

### Fully Qualified Name Resolution (NOTE: thư viện lỗi)

In [ ]:
# Necessary imports (where is fqn????)
from scalpel.fqn import FullyQualifiedNameInference as FQNInference